<img src="https://geo-credito-rural.github.io/_static/logo.jpeg" align="right" width="64" />

# <span style="color:#336699">Impedimentos Sociais, Ambientais e Climáticos</span>

<hr style="border:2px solid #0077b9;">
<div style="text-align: left;">
    <a href="https://nbviewer.jupyter.org/github/brazil-data-cube/code-gallery/blob/master/jupyter/Python/stac/stac-image-processing.ipynb"><img src="https://raw.githubusercontent.com/jupyter/design/master/logos/Badges/nbviewer_badge.svg" align="center"/></a>
</div>

<br/>

<div style="text-align: center;font-size: 90%;">
     Gabriel Sansigolo, Thales Sehn Körting, Gilberto Queiroz, Karine Ferreira, Marcos Adami
    <br/><br/>
    Divisão de Observação da Terra e Geoinformática, Instituto Nacional de Pesquisas Espaciais (INPE)
    <br/>
    Avenida dos Astronautas, 1758, Jardim da Granja, São José dos Campos, SP 12227-010, Brazil
    <br/><br/>
    Contato: <a href="https://geo-credito-rural.github.io/">Equipe - Geo Credito Rural</a>
    <br/><br/>
    Última atualização: 11 de abril de 2026
</div>


<br/>

</div>

Esse notebook aborda de forma simplificada as restrições legais para o acesso ao crédito rural com base em critérios sociais, ambientais e climáticos.

# <span style="color:#336699">Contextualização</span>
<hr style="border:1px solid #0077b9;">

Com as recentes atualizações do Conselho Monetário Nacional (como as Resoluções CMN Nº 5.193/2024, 5.267/2025 e 5.268/2025), a verificação de financiamentos tornou-se mais rigorosa. Hoje, exige-se a identificação da propriedade via Cadastro Ambiental Rural (CAR) e o monitoramento obrigatório por sensoriamento remoto para áreas superiores a 300 hectares, garantindo que os recursos financiados respeitem a conformidade socioambiental.

Na atividade prática de hoje, vamos analisar os impedimentos definidos para esses empreendimentos.

# <span style="color:#336699">8 - Desmatamento</span>
<hr style="border:1px solid #0077b9;">

O Projeto de Monitoramento do Desmatamento da Floresta Amazônica Brasileira por Satélite (PRODES) teve uma evolução na metodologia para mapeamento do desmatamento tanto por conta da evolução tecnológica quanto pelo entendimento dessa classe. Cabe ressaltar que até o ano de 2021 os polígonos de desmatamento mapeados pelo PRODES eram agregados em uma classe única “desmatamento”. Somente a partir de 2022, as classes “desmatamento por corte raso” e “desmatamento por degradação progressiva” foram incluídas nos arquivos vetoriais disponibilizados. A classe “desmatamento por corte raso” agrega todos os polígonos onde a remoção completa da floresta ocorreu em um curto período, entre o ano anterior e o ano corrente. Na imagem do ano corrente, as feições de desmatamento por corte raso podem ser detectadas com solo exposto, com vegetação herbácea, com queimada ou como mineração. Nas imagens dos anos anteriores estas áreas em questão se apresentam cobertas com floresta pouco ou nada degradada.

De acordo com a Resolução CMN Nº 5.193/2024:

> 17 - A instituição financeira deve verificar se houve supressão da vegetação nativa após 31 de julho de 2019 no
imóvel rural onde será conduzido o empreendimento, por meio de consulta às informações obtidas e
disponibilizadas pelo Ministério do Meio Ambiente e Mudança do Clima, a partir da base de dados do Projeto de
Monitoramento do Desmatamento na Amazônia Legal por Satélite – PRODES do Instituto Nacional de
Pesquisas Espaciais, observando-se que essa exigência terá início em: (Res CMN 5.268 art 1º) (*)
a) 1º de abril de 2026, quando se tratar de imóveis com área superior a quatro módulos fiscais; e
b) 4 de janeiro de 2027, quando se tratar de imóveis com área de até quatro módulos fiscais

> 18 - Caso tenha sido constatada supressão da vegetação nativa na forma do item 17, a concessão de crédito rural
com recursos controlados, conforme o MCR 6-1-2, e com recursos direcionados, conforme o MCR 6-7-7-“a”,
fica condicionada à apresentação pelo mutuário de um dos seguintes documentos referentes à supressão
constatada no imóvel, que integrarão o dossiê da operação: (Res CMN 5.193 art 1º) a) Autorização de Supressão de Vegetação (ASV) ou Autorização para Uso Alternativo do Solo (UAS)
relacionada à área desmatada após 31 de julho de 2019, conforme art. 26 da Lei nº 12.651, de 25 de maio de
2012;
b) documento que comprove que tenha executado ou esteja em execução o Projeto de Recuperação de Área
Degradada ou Área Alterada (PRAD) ou Termo de Compromisso do Programa de Regularização Ambiental
(PRA), aprovado pelo órgão ambiental competente;
c) Termo de Ajustamento de Conduta (TAC) firmado com o Ministério Público para regularização ambiental; ou
d) laudo técnico de sensoriamento remoto, sob responsabilidade da instituição financeira, comprovando a
ausência de desmatamento no imóvel rural após 31 de julho de 2019

> 19 - O contrato de crédito rural deve prever que, caso seja verificado o descumprimento de quaisquer obrigações
previstas nesta seção durante a vigência do financiamento, a operação poderá ser desclassificada na forma do
MCR 2-8. (Res CMN 5.268 art 1º)

Uma representação ilustrativa em forma de diagrama da restrição pode ser observada na Figura 8:

![car](https://i.imgur.com/yieY2DQ.png "Desmatamento")

Prática: vamos simular a verificação de sobreposição.




In [ ]:
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
import requests, zipfile, io
import geopandas as gpd

**Configuração e Exemplo que NÂO atende a norma**

In [ ]:
empreendimento = {
    "geometria": Polygon([(1, 1), (4, 1), (4, 4), (1, 4)]),
    "preve_desmatamento": False,
    "autorizacao_legal_desmatamento": False,
    "projeto_recuperacao_apresentado": False
}

In [ ]:
lista_desmatamento_prodes = [
    {"nome": "Polígono_01", "geometria": Polygon([(0, 0), (2, 0), (2, 2), (0, 2)])},
    {"nome": "Polígono_02", "geometria": Polygon([(5, 5), (6, 5), (6, 6), (5, 6)])},
    {"nome": "Polígono_03", "geometria": Polygon([(10, 10), (12, 10), (12, 12), (10, 12)])},
    {"nome": "Polígono_04", "geometria": Polygon([(20, 20), (25, 20), (25, 25), (20, 25)])}
]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

x_emp, y_emp = empreendimento["geometria"].exterior.xy

ax.plot(x_emp, y_emp, color='#1f77b4', linewidth=2, label='Empreendimento')
ax.fill(x_emp, y_emp, color='#1f77b4', alpha=0.3)

for i, prodes in enumerate(lista_desmatamento_prodes):
    x_prodes, y_prodes = prodes["geometria"].exterior.xy

    label = 'Desmatamento PRODES' if i == 0 else None

    ax.plot(x_prodes, y_prodes, color='#d62728', linewidth=2, label=label)
    ax.fill(x_prodes, y_prodes, color='#d62728', alpha=0.4)

ax.set_aspect('equal')
ax.set_title('Visualização: Empreendimento vs. Desmatamento PRODES')
ax.set_xlabel('Longitude / X')
ax.set_ylabel('Latitude / Y')
ax.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
imovel_atende_norma = True
motivo_impedimento = ""

# Verifica se o plano do empreendimento já prevê a supressão de vegetação nativa.
if empreendimento["preve_desmatamento"]:
    imovel_atende_norma = False
    motivo_impedimento = "o projeto do empreendimento prevê desmatamento."
else:

    # Inicia a comparação com a base de dados de desmatamentos já ocorridos.
    for i, desmate in enumerate(lista_desmatamento_prodes):

        # Verifica se a geometria do empreendimento possui sobreposição com algum polígono de desmatamento.
        if empreendimento["geometria"].intersects(desmate["geometria"]):

            # Verifica se o produtor comprovou que o desmatamento foi autorizado por lei.
            if empreendimento["autorizacao_legal_desmatamento"]:

                # Verifica se o produtor apresentou o projeto de recuperação da área degradada.
                if empreendimento["projeto_recuperacao_apresentado"]:
                    pass
                else:
                    imovel_atende_norma = False
                    motivo_impedimento = f"falta de projeto de recuperação para o desmatamento no {desmate['nome']}."
                    break

            # Caso o desmatamento detectado não possua autorização legal.
            else:
                imovel_atende_norma = False
                motivo_impedimento = f"desmatamento sem autorização legal detectado no {desmate['nome']}."
                break

        # Caso não haja sobreposição com polígonos de desmatamento.
        else:
            pass

# Verifica se o imóvel passou por todas as fases de checagem ambiental.
if imovel_atende_norma:
    print("O empreendimento atende a norma.")
# Exibe o motivo do impedimento relacionado ao desmatamento.
else:
    print(f"O empreendimento não atende a norma porque: {motivo_impedimento}")

**Configuração e Exemplo atende a norma**

In [ ]:
empreendimento = {
    "geometria": Polygon([(1, 1), (4, 1), (4, 4), (1, 4)]),
    "preve_desmatamento": False,
    "autorizacao_legal_desmatamento": True,
    "projeto_recuperacao_apresentado": True
}

In [ ]:
lista_desmatamento_prodes = [
    {"nome": "Polígono_01", "geometria": Polygon([(0, 0), (2, 0), (2, 2), (0, 2)])},
    {"nome": "Polígono_03", "geometria": Polygon([(10, 10), (12, 10), (12, 12), (10, 12)])},
    {"nome": "Polígono_04", "geometria": Polygon([(20, 20), (25, 20), (25, 25), (20, 25)])}
]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

x_emp, y_emp = empreendimento["geometria"].exterior.xy

ax.plot(x_emp, y_emp, color='#1f77b4', linewidth=2, label='Empreendimento')
ax.fill(x_emp, y_emp, color='#1f77b4', alpha=0.3)

for i, prodes in enumerate(lista_desmatamento_prodes):
    x_prodes, y_prodes = prodes["geometria"].exterior.xy

    label = 'Desmatamento PRODES' if i == 0 else None

    ax.plot(x_prodes, y_prodes, color='#d62728', linewidth=2, label=label)
    ax.fill(x_prodes, y_prodes, color='#d62728', alpha=0.4)

ax.set_aspect('equal')
ax.set_title('Visualização: Empreendimento vs. Desmatamento PRODES')
ax.set_xlabel('Longitude / X')
ax.set_ylabel('Latitude / Y')
ax.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
imovel_atende_norma = True
motivo_impedimento = ""

# Verifica se o plano do empreendimento já prevê a supressão de vegetação nativa.
if empreendimento["preve_desmatamento"]:
    imovel_atende_norma = False
    motivo_impedimento = "o projeto do empreendimento prevê desmatamento."
else:

    # Inicia a comparação com a base de dados de desmatamentos já ocorridos.
    for i, desmate in enumerate(lista_desmatamento_prodes):

        # Verifica se a geometria do empreendimento possui sobreposição com algum polígono de desmatamento.
        if empreendimento["geometria"].intersects(desmate["geometria"]):

            # Verifica se o produtor comprovou que o desmatamento foi autorizado por lei.
            if empreendimento["autorizacao_legal_desmatamento"]:

                # Verifica se o produtor apresentou o projeto de recuperação da área degradada.
                if empreendimento["projeto_recuperacao_apresentado"]:
                    pass
                else:
                    imovel_atende_norma = False
                    motivo_impedimento = f"falta de projeto de recuperação para o desmatamento no {desmate['nome']}."
                    break

            # Caso o desmatamento detectado não possua autorização legal.
            else:
                imovel_atende_norma = False
                motivo_impedimento = f"desmatamento sem autorização legal detectado no {desmate['nome']}."
                break

        # Caso não haja sobreposição com polígonos de desmatamento.
        else:
            pass

# Verifica se o imóvel passou por todas as fases de checagem ambiental.
if imovel_atende_norma:
    print("O empreendimento atende a norma.")
# Exibe o motivo do impedimento relacionado ao desmatamento.
else:
    print(f"O empreendimento não atende a norma porque: {motivo_impedimento}")

**Exemplo com dados reais**

**Configuração e Exemplo Não atende a norma**

In [ ]:
r = requests.get("https://raw.githubusercontent.com/GSansigolo/shapefiles/refs/heads/main/mini_prodes.zip")

zipfile.ZipFile(io.BytesIO(r.content)).extractall("./mini_prodes.zip")
file_path = "./mini_prodes.zip/mini_prodes.shp"

mini_prodes = gpd.read_file(file_path)

In [ ]:
r = requests.get("https://raw.githubusercontent.com/GSansigolo/shapefiles/refs/heads/main/poligono_16.zip")

zipfile.ZipFile(io.BytesIO(r.content)).extractall("./poligono_16.zip")
file_path = "./poligono_16.zip/poligono_16.shp"

poligono_16 = gpd.read_file(file_path)

In [ ]:
mini_prodes.tail()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(8, 8))

mini_prodes.plot(ax=ax, color='#d62728', alpha=0.3)
mini_prodes.boundary.plot(ax=ax, color='#d62728', linewidth=2)

poligono_16.plot(ax=ax, color='#1f77b4', alpha=0.3)
poligono_16.boundary.plot(ax=ax, color='#1f77b4', linewidth=2)

ax.set_aspect('equal')
ax.set_title('Visualização: Empreendimento vs. Desmatamento PRODES')
ax.set_xlabel('Longitude / X')
ax.set_ylabel('Latitude / Y')
plt.grid(True, linestyle='--', alpha=0.6)

legenda_empreendimento = mpatches.Patch(color='#1f77b4', alpha=0.5, label='Empreendimento')
legenda_tis = mpatches.Patch(color='#d62728', alpha=0.5, label='Desmatamento PRODES')
ax.legend(handles=[legenda_empreendimento, legenda_tis])
plt.show()

In [ ]:
import folium


for col in mini_prodes.select_dtypes(include=['datetime64', 'datetimetz', 'datetime64[ns]']).columns:
    mini_prodes[col] = mini_prodes[col].astype(str)

for col in poligono_16.select_dtypes(include=['datetime64', 'datetimetz', 'datetime64[ns]']).columns:
    poligono_16[col] = poligono_16[col].astype(str)


bounds = poligono_16.total_bounds
centro_y = (bounds[1] + bounds[3]) / 2
centro_x = (bounds[0] + bounds[2]) / 2

m = folium.Map(location=[centro_y, centro_x], zoom_start=13, tiles='CartoDB positron')

folium.GeoJson(
    mini_prodes,
    name='Desmatamento PRODES',
    style_function=lambda x: {
        'fillColor': '#d62728',
        'color': '#d62728',
        'weight': 2,
        'fillOpacity': 0.3
    }
).add_to(m)

folium.GeoJson(
    poligono_16,
    name='Empreendimento',
    style_function=lambda x: {
        'fillColor': '#1f77b4',
        'color': '#1f77b4',
        'weight': 2,
        'fillOpacity': 0.3
    }
).add_to(m)

folium.LayerControl().add_to(m)
m

In [ ]:
poligono_16 = poligono_16.to_crs(mini_prodes.crs)
geom_empreendimento = poligono_16.geometry.iloc[0]

mini_prodes_plano = mini_prodes.to_crs(epsg=6933)
geom_empreendimento_plana = poligono_16.to_crs(epsg=6933).geometry.iloc[0]

sobreposicoes = mini_prodes.intersects(geom_empreendimento)
areas = mini_prodes_plano.intersection(geom_empreendimento_plana).area

area_minima = 1.0

for i, sobrepoe in sobreposicoes.items():
    if sobrepoe and areas[i] > area_minima:
        print(f"O empreendimento sobrepõe o Desmatamento {i}! Área da sobreposição: {areas[i]:.2f} m².")


**Configuração e Exemplo atende a norma**

In [ ]:
r = requests.get("https://raw.githubusercontent.com/GSansigolo/shapefiles/refs/heads/main/poligono_15.zip")

zipfile.ZipFile(io.BytesIO(r.content)).extractall("./poligono_15.zip")
file_path = "./poligono_15.zip/poligono_15.shp"

poligono_15 = gpd.read_file(file_path)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(8, 8))

mini_prodes.plot(ax=ax, color='#d62728', alpha=0.3)
mini_prodes.boundary.plot(ax=ax, color='#d62728', linewidth=2)

poligono_15.plot(ax=ax, color='#1f77b4', alpha=0.3)
poligono_15.boundary.plot(ax=ax, color='#1f77b4', linewidth=2)

ax.set_aspect('equal')
ax.set_title('Visualização: Empreendimento vs. Desmatamento PRODES')
ax.set_xlabel('Longitude / X')
ax.set_ylabel('Latitude / Y')
plt.grid(True, linestyle='--', alpha=0.6)

legenda_empreendimento = mpatches.Patch(color='#1f77b4', alpha=0.5, label='Empreendimento')
legenda_tis = mpatches.Patch(color='#d62728', alpha=0.5, label='Desmatamento PRODES')
ax.legend(handles=[legenda_empreendimento, legenda_tis])
plt.show()

In [ ]:
poligono_15 = poligono_15.to_crs(mini_prodes.crs)
geom_empreendimento = poligono_15.geometry.iloc[0]

mini_prodes_plano = mini_prodes.to_crs(epsg=6933)
geom_empreendimento_plana = poligono_15.to_crs(epsg=6933).geometry.iloc[0]

sobreposicoes = mini_prodes.intersects(geom_empreendimento)
areas = mini_prodes_plano.intersection(geom_empreendimento_plana).area

area_minima = 1.0

for i, sobrepoe in sobreposicoes.items():
    if sobrepoe and areas[i] > area_minima:
        print(f"O empreendimento sobrepõe o Desmatamento {i}! Área da sobreposição: {areas[i]:.2f} m².")
